### KNN Graph is a graph construction method where each point is connected to its k nearest neighboring points based on a distance metric (typically Euclidean distance), enabling local neighborhood learning in point cloud and graph neural networks.

### Algorithm
- Calculate euclidean distance
* Two point $P_i=(x_i,y_i,z_i)$ and $P_j=(x_j,y_j,z_j)$
$$distance=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2+(z_i-z_j)^2}$$

- Sort all point
- Select nearest k points. e.g.,k=1,2,3,....n
- Connect edge among them

### Complexity
- Brute Force KNN: O(N²)
- KD-Tree: O(N log N)
- GPU-based KNN: Fast


### Example 
| Point | Coordinates (x, y, z) |
| ----- | --------------------- |
| P₁    | (1, 2, 1)             |
| P₂    | (2, 2, 1)             |
| P₃    | (1, 3, 2)             |
| P₄    | (5, 5, 5)             |
| P₅    | (6, 5, 5)             |

k=2

d(P₁​, P₂​)=1
d(P₁, P₃)=1.414
d(P₁, P₄)=6.403
d(P₁, P₅​)=7.071

- Sort the Distances

| Neighbor | Distance  |
| -------- | --------- |
| P₂       | **1.000** |
| P₃       | **1.414** |
| P₄       | 6.403     |
| P₅       | 7.071     |

- Graph edges
P₁ → P₂
P₁ → P₃

- Final KNN Graph
* If every point find k nearest neighbous.
P₁ → {P₂, P₃}

P₂ → {P₁, P₃}

P₃ → {P₁, P₂}

P₄ → {P₅, P₃}

P₅ → {P₄, P₃}

In [10]:
import torch

In [11]:
'''
https://github.com/SEULSH/Non-corresponding-and-Topology-free-3D-Face-Expression-Transfer/blob/main/dgcnn.py
'''
def knn(x, k):

    # x.shape = batch_size, num_features, num_points
    inner = -2*torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
 
    idx = pairwise_distance.topk(k=k, dim=-1)[1]   # (batch_size, num_points, k)
    return idx

| Equation                   | Code                                |
| -------------------------- | ----------------------------------- |
| $x_i^2+y_i^2+z_i^2$        | `xx = torch.sum(x**2, dim=1)`       |
| $x_j^2+y_j^2+z_j^2$        | `xx.transpose(2,1)`                 |
| $x_ix_j+y_iy_j+z_iz_j$     | `torch.matmul(x.transpose(2,1), x)` |
| $-2(x_ix_j+y_iy_j+z_iz_j)$ | `inner = -2 * torch.matmul(...)`    |
| $\|p_i-p_j\|^2$             | `xx + xx.T - 2*dot`    |
| $-\|p_i-p_j\|^2$             | `-xx - inner - xx.T`                |
| k nearest neighbor         | `pairwise_distance.topk(k)`         |


In [12]:
x=torch.tensor([
    [
        [1,2,1], # P1
        [2,2,1], # P2
        [1,3,2], # P3
        [5,5,5], # P4
        [6,5,5]  # P5
    ]
])

* But when knn then it consider,
```txt
x=[[
    [], # x coordinates
    [], # y coordinates
    [], # z coordinates
]]
```

so, x.transpose(2,1)

In [13]:
knn(x.transpose(2,1),5)

tensor([[[0, 1, 2, 3, 4],
         [1, 0, 2, 3, 4],
         [2, 0, 1, 3, 4],
         [3, 4, 2, 1, 0],
         [4, 3, 2, 1, 0]]])